In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.format("parquet")\
               .load("abfss://bronze@olympicsprojshrishti.dfs.core.windows.net/athletes")

In [0]:
display(df)

**Applying transformations**

In [0]:
## Replacing null values in the following columns with respective default values
df = df.fillna(
    {"birth_place": "xyz",
     "birth_country": "abc",
     "residence_place": "Unknown",
     "residence_country": "jkl"})
df.display()

In [0]:
## Filtering dataframe based on multiple conditions
df_filtered = df.filter((col("current")==True) & col("name").isin("VLACH Martin", "MARLETTA Claudia Roberta","HLAVACKOVA Lucie"))
df_filtered.display()

In [0]:
## Casting columns to appropriate data types
df = df.withColumn('height', col('height').cast(FloatType()))\
        .withColumn('weight', col('weight').cast(FloatType()))
df.display()

In [0]:
# Sorting dataframe based on multiple columns
df_sorted = df.sort('height', 'weight', ascending = [0,1]).filter(col('weight') >0)
df_sorted.display()

In [0]:
# Replacing values in a column
df_sorted = df_sorted.withColumn('nationality', regexp_replace('nationality', 'United States', 'US'))
df_sorted.display()

In [0]:
# Checking for duplicates
df.groupBy('code').agg(count('code').alias('total_count')).filter(col('total_count')>1).display()

In [0]:
# Renaming Primary key column
df_sorted = df_sorted.withColumnRenamed('code', 'athlete_id')
df_sorted.display()

In [0]:
# Converting a column to a list
df_sorted = df_sorted.withColumn('occupation', split('occupation', ','))
df_sorted.display()

In [0]:
df_sorted.columns

In [0]:
# Pruning our data
df_final = df_sorted.select('athlete_id',
 'current',
 'name',
 'name_short',
 'name_tv',
 'gender',
 'function',
 'country_code',
 'country',
 'country_long',
 'nationality',
 'nationality_long',
 'nationality_code',
 'height',
 'weight')

In [0]:
# Displaying our data and building basic visualisation
display(df_final)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.window import Window

In [0]:
# Calculating cumulative sum of weights on nationality column
df_final = df_final.withColumn("cum_weight", sum("weight").over(Window.partitionBy("nationality").orderBy("height").rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)))
df_final.display()

In [0]:
df_final.createOrReplaceTempView("df_final")

In [0]:
## Performing same transformation in SQL
result_df = spark.sql("""
          SELECT nationality, height, weight, sum(weight) over(PARTITION BY nationality order by height rows BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) as total_weight FROM df_final
          """)
result_df.display()

**Writing the data**

In [0]:
# Writing as an external delta table
df_final.write.format("delta")\
            .mode("append")\
            .option("path", "abfss://silver@olympicsprojshrishti.dfs.core.windows.net/athletes")\
            .saveAsTable("olympics.silver.athletes")